# BRECHA CATASTRAL Y ASENTAMIENTOS INFORMALES · HONDURAS
## Kit Replicable para Gobiernos Locales y Gestión Territorial
### Clústers Copernicus Centroamérica – Fase II · Ficha N.° 8

**Zona Piloto:** Mancha urbana consolidada del Municipio del Distrito Central (Tegucigalpa y Comayagüela)

**Equipo Técnico:**
- **Mario Baide** · Universidad Nacional Autónoma de Honduras (UNAH) — *Profesor Investigador*
- **Mario Matamoros** · Universidad Nacional Autónoma de Honduras (UNAH) — *Profesor Investigador*
- **Jonathan Ramirez** · Alcaldía Municipal del Distrito Central (AMDC) — *Analista en Planificación Urbana*
- **Ana Maritza Irías** · Secretaría de Gobernación, Justicia y Descentralización (SEGOB) — *Especialista institucional*

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariobaidemunoz-del/urbano-hn-distrito-central/blob/main/notebooks/brecha_catastral_distrito_central.ipynb)

## 1. Instalación de librerías y dependencias
Instalamos `geemap`, `earthengine-api`, `geopandas` y herramientas de análisis espacial para Google Colab.

In [ ]:
!pip install -q geemap earthengine-api geopandas matplotlib

## 2. Autenticación e Inicialización de Google Earth Engine
Ejecuta la siguiente celda e introduce tus credenciales de Google cuando se solicite.

In [ ]:
import ee
import geemap
import math
import json
import geopandas as gpd
import matplotlib.pyplot as plt

# Proyecto de Google Cloud registrado en Earth Engine (obligatorio desde 2024).
# Es el mismo proyecto donde se suben los assets privados de la AMDC.
EE_PROJECT = 'proyecto-honduras-514315'   # proyecto del equipo MDC; para replicar, usar el propio

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print(f"✔ Google Earth Engine inicializado en el proyecto {EE_PROJECT}.")


## 3. Parámetros de Configuración (`CONFIG`)
Permite parametrizar el análisis para el **Distrito Central** o replicarlo en cualquier municipio de Honduras.

In [ ]:
CONFIG = {
    'pais': 'HND',
    'municipio': 'Distrito Central',      # Nombre en geoBoundaries ADM2
    'ambito': 'perimetro',                 # 'perimetro' = PU 2025 AMDC · 'mancha' = GHSL consolidada · 'municipio'
    'umbralManchaM2': 1000,                # m² construidos por celda GHSL de 1 ha para considerarla urbana
    'areaMinManchaHa': 50,                 # Descarta núcleos aislados menores a 50 ha
    'anioInicio': 2016,
    'anioFin': 2026,
    'mesesSecos': [1, 4],                  # Enero a Abril (estación seca)
    'confianzaEdif': 0.70,                 # Umbral de confianza Open Buildings v3
    'umbralPresenciaOB': 0.50,             # Umbral building_presence (2.5D Temporal)
    'pendienteCritica': 30,                # % de pendiente considerada crítica
    'tamCelda': 250,                       # Retícula de análisis (metros)
    'epsg': 'EPSG:32616',                  # UTM 16N (Honduras)
    'densidadInformal': 40,                # Edificaciones por hectárea
    'areaMediaInformal': 80,               # m² medios por edificación
    'umbralLadera': 0.30,                  # Proporción en pendiente crítica o riesgo
    'umbralNuevo': 0.50,                   # Proporción edificación aparecida tras año base
    'umbralFueraCatastro': 0.50,           # Proporción de m² construidos fuera de predio catastrado
    'umbralLoteOcupado': 0.10,             # Fracción del lote cubierta por edificación para considerarlo ocupado
    # Unidad de agregación pública. 'celda' (250 m) es la que mide la brecha: manzanas y colonias
    # SON catastro, así que agregando por ellas lo construido fuera de catastro queda oculto.
    'unidad': 'celda',                     # 'celda' · 'manzana' · 'colonia'
    # Etapa 1: guarda en assets una imagen de 4 m con lo construido y lo catastrado (una sola vez).
    # Etapa 2: agrega esa imagen por celda o manzana (rápido). Pasar a 2 cuando termine la etapa 1.
    'etapa': 1,
    'assetBase': f'projects/{EE_PROJECT}/assets/mdc_base_4m',
    'cortesCelda': [-87.30, -87.2577, -87.2268, -87.1959, -87.1650, -87.1341, -87.09],  # franjas O–E para exportar
    # Lo catastrado: assets PRIVADOS de la AMDC (no se publican; ver sección 3b)
    'assetPredios':   f'projects/{EE_PROJECT}/assets/amdc_predios_2026',
    'assetManzanas':  f'projects/{EE_PROJECT}/assets/amdc_manzanas_2026',
    'assetColonias':  f'projects/{EE_PROJECT}/assets/amdc_colonias_2026',
    'assetPerimetro': f'projects/{EE_PROJECT}/assets/amdc_pu_2026'
}
print(f"Configuración cargada para {CONFIG['municipio']}, {CONFIG['pais']}.")

## 3b. Lo catastrado: planimetría de la AMDC como assets privados
El parcelario se usa **solo para el cálculo**; nunca se publica a nivel de lote. Antes de ejecutar, sube a Earth Engine (Code Editor → *Assets* → *New* → *Shape files*) los cuatro zip preparados en `AMDC_para_GEE/`, con estos nombres de asset:

| Archivo | Asset | Contenido |
|---|---|---|
| `amdc_predios_2026.zip` | `amdc_predios_2026` | Linderos de predio (SECTOR, MANZANA, LOTE, CLAVE_CAT, COD_COL, AREA_M2) |
| `amdc_manzanas_2026.zip` | `amdc_manzanas_2026` | Linderos de manzana |
| `amdc_colonias_2026.zip` | `amdc_colonias_2026` | Colonias del área urbana |
| `amdc_pu_2026.zip` | `amdc_pu_2026` | Perímetro urbano PU 2025 |

Comparte los assets solo con el equipo técnico (UNAH · AMDC · SEGOB).


In [ ]:
def cargar_catastro(config):
    predios = ee.FeatureCollection(config['assetPredios'])
    catastrado = ee.Image(0).byte().paint(predios, 1).rename('catastrado')
    return predios, catastrado

try:
    predios, catastrado = cargar_catastro(CONFIG)
    print(f"✔ Predios AMDC cargados: {predios.size().getInfo():,}")
except Exception as e:
    print("⚠ No se encontraron los assets de la AMDC. Revisa EE_PROJECT y la sección 3b.\n", e)


## 4. Delimitación Territorial
Se distinguen dos polígonos que **no son equivalentes**:
- **Perímetro urbano oficial (PU 2025, AMDC):** ámbito administrativo de *lo catastrado*; incluye áreas no construidas.
- **Mancha urbana consolidada (GHSL 2020):** huella de *lo construido*, por cierre morfológico de la superficie construida.

`CONFIG['ambito']` decide sobre cuál se calculan los indicadores; la mancha GHSL se exporta siempre como capa del geovisor.


In [ ]:
def obtener_municipio(nombre, pais='HND'):
    adm2 = ee.FeatureCollection('WM/geoLab/geoBoundaries/600/ADM2').filter(ee.Filter.eq('shapeGroup', pais))
    return adm2.filter(ee.Filter.eq('shapeName', nombre)).geometry()

def mancha_ghsl(muni, config):
    """Lo construido: mancha urbana consolidada a partir de GHSL 2020 (100 m)."""
    ghsl = ee.Image(ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S')
                    .filterDate('2020-01-01', '2021-01-01').first()).select('built_surface').clip(muni)
    urbano = ghsl.gte(config['umbralManchaM2'])\
                 .focalMax(1, 'square', 'pixels')\
                 .focalMin(1, 'square', 'pixels')\
                 .selfMask()
    poligonos = urbano.reduceToVectors(
        geometry=muni, scale=100, crs=config['epsg'], geometryType='polygon',
        eightConnected=True, labelProperty='urbano', maxPixels=1e10
    )
    filtrados = poligonos.map(lambda f: f.set('ha', f.geometry().area(10).divide(10000)))\
                         .filter(ee.Filter.gte('ha', config['areaMinManchaHa']))
    return filtrados.geometry(10)

def obtener_ambito(nombre, config):
    muni = obtener_municipio(nombre, config['pais'])
    if config.get('ambito') == 'municipio':
        return muni
    if config.get('ambito') == 'perimetro':
        return ee.FeatureCollection(config['assetPerimetro']).geometry()
    return mancha_ghsl(muni, config)

muni = obtener_municipio(CONFIG['municipio'], CONFIG['pais'])
mancha = mancha_ghsl(muni, CONFIG)
aoi = obtener_ambito(CONFIG['municipio'], CONFIG)
print(f"✔ Ámbito de análisis: {CONFIG['ambito']}. Mancha GHSL: {mancha.area(100).divide(1e4).getInfo():,.0f} ha")


## 5. Modelado de la "Ciudad Construida"
- **Sentinel-2 L2A**: Compuesto de mediana de estación seca con máscara Cloud Score+.
- **Índices espectrales**: NDBI (construido), NDVI (vegetación) y BSI (suelo desnudo).
- **Dynamic World**: Detección de consolidación urbana (2016–2026).
- **GHSL**: Evolución histórica a largo plazo (1975–2020).
- **Open Buildings v3 y 2.5D Temporal**: Huellas individuales y año de aparición (2016–2023).

In [ ]:
def composite_s2(aoi, anio, config):
    cs = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    inicio = ee.Date.fromYMD(anio, config['mesesSecos'][0], 1)
    fin = ee.Date.fromYMD(anio, config['mesesSecos'][1], 1).advance(1, 'month')
    
    def mascara_nubes(img):
        return img.updateMask(img.select('cs_cdf').gte(0.60))\
                  .select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])\
                  .divide(10000)
                  
    return ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
             .filterBounds(aoi).filterDate(inicio, fin)\
             .linkCollection(cs, ['cs_cdf'])\
             .map(mascara_nubes).median().clip(aoi)

def indices_espectrales(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndbi = img.normalizedDifference(['B11', 'B8']).rename('NDBI')
    bsi = img.expression(
        '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
        {'SWIR': img.select('B11'), 'RED': img.select('B4'), 'NIR': img.select('B8'), 'BLUE': img.select('B2')}
    ).rename('BSI')
    return ndvi.addBands(ndbi).addBands(bsi)

def construido_anual(aoi, anio, config):
    inicio = ee.Date.fromYMD(anio, config['mesesSecos'][0], 1)
    fin = ee.Date.fromYMD(anio, config['mesesSecos'][1], 1).advance(1, 'month')
    return ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')\
             .filterBounds(aoi).filterDate(inicio, fin)\
             .select('label').mode().eq(6)\
             .rename('construido').set('anio', anio).clip(aoi)

def anio_consolidacion(aoi, config):
    anios = ee.List.sequence(config['anioInicio'], config['anioFin'] - 1)
    def consolidar_ano(a):
        a = ee.Number(a)
        hoy = construido_anual(aoi, a, config)
        sig = construido_anual(aoi, a.add(1), config)
        estable = hoy.And(sig)
        return ee.Image.constant(a).toInt16().updateMask(estable).rename('anio')
    imgs = anios.map(consolidar_ano)
    return ee.ImageCollection.fromImages(imgs).min().clip(aoi)

def historia_ghsl(aoi):
    ghsl = ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S')
    epocas = [1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020]
    def mapear_epoca(e):
        e = ee.Number(e)
        d1 = ee.Date.fromYMD(e, 1, 1)
        d2 = ee.Date.fromYMD(e.add(1), 1, 1)
        img = ee.Image(ghsl.filterDate(d1, d2).first()).select('built_surface')
        return ee.Image.constant(e).toInt16().updateMask(img.gte(1000)).rename('epoca')
    imgs = ee.List(epocas).map(mapear_epoca)
    return ee.ImageCollection.fromImages(imgs).min().clip(aoi)

def edificaciones(aoi, config):
    return ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons')\
             .filterBounds(aoi).filter(ee.Filter.gte('confidence', config['confianzaEdif']))

def anio_aparicion_edif(aoi, config):
    ob = ee.ImageCollection('GOOGLE/Research/open-buildings-temporal/v1').filterBounds(aoi)
    anios = ee.List.sequence(2016, 2023)
    def mapear_ob(a):
        a = ee.Number(a)
        anual = ob.filter(ee.Filter.calendarRange(a, a, 'year')).mosaic()
        presente = anual.select('building_presence').gte(config['umbralPresenciaOB'])
        return ee.Image.constant(a).toInt16().updateMask(presente).rename('anio_edif')
    imgs = anios.map(mapear_ob)
    return ee.ImageCollection.fromImages(imgs).min().clip(aoi)

print("✔ Funciones de ciudad construida definidas.")

## 6. Condicionantes Críticos y Pendiente (Copernicus DEM GLO-30)
Cálculo de pendientes en porcentaje a partir del Modelo Digital de Elevación de Copernicus a 30 metros de resolución.

In [ ]:
def condicionantes(aoi, config):
    dem_col = ee.ImageCollection('COPERNICUS/DEM/GLO30').filterBounds(aoi).select('DEM')
    dem = dem_col.mosaic().setDefaultProjection(dem_col.first().projection())
    pendiente_grados = ee.Terrain.slope(dem)
    pendiente_pct = pendiente_grados.multiply(math.pi / 180).tan().multiply(100)
    critica = pendiente_pct.gte(config['pendienteCritica'])
    return critica.rename('condicion_critica').clip(aoi)

print("✔ Condicionantes de pendiente crítica definidos.")

## 7. Cruce catastrado vs. construido, indicadores y tipología
Para cada unidad de análisis (`CONFIG['unidad']`; por defecto, celdas de 250 m recortadas al ámbito) se calculan:
- **Lo construido:** m² y número de edificaciones (Open Buildings v3), m² nuevos desde 2016, m² en pendiente crítica.
- **Lo catastrado:** cobertura de predios y número de lotes (la ocupación por lote se exporta aparte, para uso interno).
- **La brecha:** edificaciones y m² construidos **fuera de predio catastrado**.

La clasificación `tipologia_brecha` es una combinación preliminar (brecha × morfología) para revisión del equipo; **no reemplaza la tipología A–E**, cuyas reglas deben documentarse antes de asignarla.


> **Por qué celdas y no manzanas:** en una prueba con 1.998 manzanas de los sectores centrales, solo el 1 % de lo construido quedó fuera de catastro, porque dentro de una manzana casi todo el suelo tiene predio. La construcción sin catastro está sobre todo *fuera* de las manzanas dibujadas, y solo una unidad independiente del catastro (la celda) la capta.


In [ ]:
def unidades_de_analisis(aoi, config):
    u = config.get('unidad', 'celda')
    if u == 'manzana':
        fc = ee.FeatureCollection(config['assetManzanas']).filterBounds(aoi)
    elif u == 'colonia':
        fc = ee.FeatureCollection(config['assetColonias']).filterBounds(aoi)
    else:
        # Celdas completas: recortarlas aquí al ámbito multiplica el tiempo de cálculo. La superficie
        # válida sale de los píxeles dentro del ámbito (m2_valido) y el recorte geométrico se hace al
        # integrar la capa al geovisor (src/gee/unir_celdas.py). Se exportan en UTM 16N.
        proj = ee.Projection(config['epsg']).atScale(config['tamCelda'])
        return aoi.coveringGrid(proj).map(
            lambda f: f.set({'unidad': f.id(), 'cx': f.geometry().centroid(1).coordinates().get(0)}))
    return fc.map(lambda f: f.set('unidad', f.id()))

def centroides(fc):
    return fc.map(lambda f: ee.Feature(f.centroid(1)))

def contar(fc):
    """Pinta el píxel que contiene el centroide de cada elemento (más rápido que reduceToImage)."""
    return ee.Image(0).byte().paint(centroides(fc), 1)

def ocupacion_lotes(predios, huella_edif, config):
    """Nivel lote (USO INTERNO): m² construidos dentro de cada predio."""
    m2 = huella_edif.multiply(ee.Image.pixelArea()).rename('m2_edif')
    lotes = m2.reduceRegions(collection=predios, reducer=ee.Reducer.sum().setOutputs(['m2_edif']),
                             scale=2, crs=config['epsg'], tileScale=4)
    def marcar(f):
        a = f.geometry().area(1).max(1)
        p = ee.Number(f.get('m2_edif')).divide(a)
        return f.set({'p_ocupado': p, 'ocupado': p.gte(config['umbralLoteOcupado'])})
    return lotes.map(marcar)

def calcular_indicadores(aoi, config):
    # Lo construido
    edif = edificaciones(aoi, config)
    huella_edif = ee.Image(0).byte().paint(edif, 1).rename('edif').clip(aoi)
    aparicion = anio_aparicion_edif(aoi, config)
    nuevo = aparicion.gte(config['anioInicio'] + 1).unmask(0).And(huella_edif).rename('edif_nueva')
    critica = condicionantes(aoi, config)
    conteo = contar(edif).rename('n_edif')

    # Lo catastrado
    predios = ee.FeatureCollection(config['assetPredios']).filterBounds(aoi)
    catastrado = ee.Image(0).byte().paint(predios, 1).rename('catastrado')
    n_lotes = contar(predios)

    # La brecha
    fuera = huella_edif.And(catastrado.Not()).rename('edif_fuera')

    # Imagen base de 4 m (0/1 por banda). Pintar cientos de miles de polígonos en cada agregación
    # excede el tiempo de Earth Engine; por eso se materializa una vez como asset (etapa 1).
    base = huella_edif.rename('edif')\
        .addBands(nuevo.rename('nuevo'))\
        .addBands(critica.rename('critica'))\
        .addBands(conteo.rename('conteo'))\
        .addBands(catastrado.rename('catastrado'))\
        .addBands(n_lotes.rename('lotes'))\
        .toByte().clip(aoi)
    B = ee.Image(config['assetBase']) if config.get('etapa') == 2 else base

    area = ee.Image.pixelArea()
    pila = B.select('edif').multiply(area).rename('m2_edif')\
        .addBands(B.select('nuevo').multiply(area).rename('m2_nuevo'))\
        .addBands(B.select('edif').And(B.select('critica')).multiply(area).rename('m2_critico'))\
        .addBands(B.select('conteo').rename('n_edif'))\
        .addBands(B.select('catastrado').multiply(area).rename('m2_catastrado'))\
        .addBands(B.select('edif').And(B.select('catastrado').Not()).multiply(area).rename('m2_fuera'))\
        .addBands(B.select('conteo').multiply(B.select('catastrado').Not()).rename('n_edif_fuera'))\
        .addBands(B.select('lotes').rename('n_lotes'))\
        .addBands(B.select('edif').mask().multiply(area).rename('m2_valido'))

    def indicadores(unidades):
        return pila.reduceRegions(collection=unidades, reducer=ee.Reducer.sum(),
                                  scale=4, crs=config['epsg'], tileScale=4).map(tipificar)

    def tipificar(f):
        ha = ee.Number(f.get('m2_valido')).divide(10000).max(0.0001)   # superficie dentro del ámbito
        m2 = ee.Number(f.get('m2_edif'))
        hay = m2.gt(0)
        den = m2.max(1)
        n = ee.Number(f.get('n_edif'))
        p_nuevo = ee.Number(f.get('m2_nuevo')).divide(den)
        p_critico = ee.Number(f.get('m2_critico')).divide(den)
        p_fuera = ee.Number(f.get('m2_fuera')).divide(den)
        densidad = n.divide(ha)
        area_media = m2.divide(n.max(1))
        morf_informal = densidad.gte(config['densidadInformal']).And(area_media.lte(config['areaMediaInformal']))
        brecha = p_fuera.gte(config['umbralFueraCatastro'])

        tipo = ee.Algorithms.If(hay.Not(), 'Sin edificación',
               ee.Algorithms.If(morf_informal.And(p_critico.gte(config['umbralLadera'])), 'Candidato informal en condición crítica',
               ee.Algorithms.If(morf_informal.And(p_nuevo.gte(config['umbralNuevo'])), 'Candidato informal reciente',
               ee.Algorithms.If(morf_informal, 'Candidato informal consolidado',
               'Tejido no clasificado como informal'))))
        tipo_brecha = ee.Algorithms.If(hay.Not(), 'Sin edificación',
                      ee.Algorithms.If(brecha.And(morf_informal), 'Brecha catastral + morfología informal',
                      ee.Algorithms.If(brecha, 'Brecha catastral',
                      ee.Algorithms.If(morf_informal, 'Morfología informal dentro de catastro',
                      'Construido dentro de catastro'))))

        props = {
            'ha': ha,
            'n_edif': n,
            'dens_edif_ha': densidad,
            'area_media_m2': area_media,
            'cobertura_edif': m2.divide(ha.multiply(10000)),
            'p_nuevo': p_nuevo,
            'p_critico': p_critico,
            'cobertura_catastral': ee.Number(f.get('m2_catastrado')).divide(ha.multiply(10000)),
            'p_fuera_catastro': p_fuera,
            'morf_informal': morf_informal,
            'brecha_catastral': brecha,
            'tipologia': tipo,
            'tipologia_brecha': tipo_brecha
        }
        return f.set(props)

    unidades = unidades_de_analisis(aoi, config)
    tabla = indicadores(unidades)
    return {
        'edif': edif, 'huella_edif': huella_edif, 'aparicion': aparicion,
        'nuevo': nuevo, 'critica': critica, 'tabla': tabla,
        'predios': predios, 'catastrado': catastrado, 'fuera': fuera,
        'unidades': unidades, 'indicadores': indicadores,
        'base': base,
        'lotes': lambda filtro: ocupacion_lotes(predios.filter(filtro), B.select('edif'), config)
    }

print("✔ Módulo de cruce catastrado vs. construido configurado.")


## 8. Ejecución del Procesamiento Espacial
Generamos las capas analíticas para la zona del Distrito Central.

In [ ]:
print("Procesando capas multitemporales para el Distrito Central...")
s2_composite = composite_s2(aoi, CONFIG['anioFin'], CONFIG)
idx = indices_espectrales(s2_composite)
consol = anio_consolidacion(aoi, CONFIG)
ghsl = historia_ghsl(aoi)
resultado = calcular_indicadores(aoi, CONFIG)
print("✔ Capas y tabla de indicadores generadas exitosamente.")

## 9. Visualizador Interactivo con `geemap`
Explora todas las capas satelitales, índices, huellas de edificaciones y la tipología de asentamientos informales directamente en el mapa.

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, 12)

pal_anios = ['#fff5eb', '#fdd0a2', '#fd8d3c', '#d94801', '#7f2704']

# 1. Imagen Sentinel-2 RGB
Map.addLayer(s2_composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.25}, 'Sentinel-2 RGB 2026')

# 2. Índice NDBI (Índice de Edificación Normalizado)
Map.addLayer(idx.select('NDBI'), {'min': -0.3, 'max': 0.3, 'palette': ['#1a9850', '#ffffbf', '#d73027']}, 'NDBI (Construido)', False)

# 3. GHSL Épocas (1975–2020)
Map.addLayer(ghsl, {'min': 1975, 'max': 2020, 'palette': pal_anios}, 'GHSL Huella Histórica (1975–2020)', False)

# 4. Consolidación Dynamic World
Map.addLayer(consol, {'min': CONFIG['anioInicio'], 'max': CONFIG['anioFin'] - 1, 'palette': pal_anios}, 'Año de Consolidación (Dynamic World)', False)

# 5. Aparición Open Buildings
Map.addLayer(resultado['aparicion'], {'min': 2016, 'max': 2023, 'palette': pal_anios}, 'Año Aparición Edificaciones (OB 2.5D)', False)

# 6. Huellas individuales de edificaciones
Map.addLayer(resultado['edif'].style(color='444444', width=0.5, fillColor='00000000'), {}, 'Edificaciones (Open Buildings v3)', False)

# 7. Condición crítica (Pendiente >= 30%)
Map.addLayer(resultado['critica'].selfMask(), {'palette': ['#6a51a3'], 'opacity': 0.45}, 'Pendiente Crítica (>= 30%)', False)

# 8. Ámbito territorial de análisis
Map.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(aoi)]), 1, 2), {'palette': ['#FFC800']}, f"Ámbito de análisis ({CONFIG['ambito']})")

# 8b. Lo catastrado y la brecha
Map.addLayer(resultado['predios'].style(color='1f78b4', width=0.4, fillColor='00000000'), {}, 'Predios AMDC (uso interno)', False)
Map.addLayer(resultado['catastrado'].selfMask(), {'palette': ['#a6cee3'], 'opacity': 0.35}, 'Superficie catastrada', False)
Map.addLayer(resultado['fuera'].selfMask(), {'palette': ['#e31a1c']}, 'Edificación fuera de predio catastrado')
Map.addLayer(ee.Image().float().paint(resultado['tabla'], 'p_fuera_catastro'),
             {'min': 0, 'max': 1, 'palette': ['#fff5f0', '#fcbba1', '#fb6a4a', '#cb181d', '#67000d']},
             f"% construido fuera de catastro por {CONFIG['unidad']}", False)
Map.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(mancha)]), 1, 2), {'palette': ['#FF7F00']}, 'Mancha GHSL 2020 (construido)', False)

# 9. Tipología de candidatos a asentamiento informal
tipos = [
    'Tejido no clasificado como informal',
    'Candidato informal consolidado',
    'Candidato informal reciente',
    'Candidato informal en condición crítica'
]
colores = ['#d9d9d9', '#f46d43', '#d73027', '#67001f']

codigo = resultado['tabla'].filter(ee.Filter.neq('tipologia', 'Sin edificación'))\
                           .map(lambda f: f.set('cod', ee.List(tipos).indexOf(f.get('tipologia'))))
img_tipo = ee.Image().byte().paint(codigo, 'cod')
Map.addLayer(img_tipo, {'min': 0, 'max': len(tipos) - 1, 'palette': colores}, 'Tipología de Asentamientos')

# Leyenda explicativa
legend_dict = dict(zip(tipos, colores))
Map.add_legend(title="Tipología Urbana", legend_dict=legend_dict)

Map

## 10. Análisis Estadístico de la Brecha y Asentamientos
Extraemos la muestra de indicadores para graficar la distribución de nueva edificación y criticidad en la mancha urbana.

In [ ]:
conteo_total = resultado['tabla'].filter(ee.Filter.neq('tipologia', 'Sin edificación')).size().getInfo()
print(f"Total de celdas de 250m con edificación analizadas: {conteo_total}")

# Extraer una muestra para graficar
muestra = resultado['tabla'].filter(ee.Filter.neq('tipologia', 'Sin edificación'))\
                            .limit(1000)\
                            .reduceColumns(ee.Reducer.toList(3), ['p_nuevo', 'p_critico', 'dens_edif_ha'])\
                            .get('list').getInfo()

if muestra:
    import pandas as pd
    df = pd.DataFrame(muestra, columns=['Proporción Nueva Edificación', 'Proporción en Pendiente Crítica', 'Densidad (edif/ha)'])
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 4))
    df['Proporción Nueva Edificación'].hist(bins=20, ax=axs[0], color='#d73027', edgecolor='black')
    axs[0].set_title('Distribución de Nueva Edificación (Post-2016)')
    axs[0].set_xlabel('Proporción')
    
    df['Proporción en Pendiente Crítica'].hist(bins=20, ax=axs[1], color='#6a51a3', edgecolor='black')
    axs[1].set_title('Exposición en Ladera Crítica (>= 30%)')
    axs[1].set_xlabel('Proporción')
    
    plt.tight_layout()
    plt.show()

# Resumen de la brecha catastral en el ámbito
tot = resultado['tabla'].reduceColumns(ee.Reducer.sum().repeat(4),
        ['n_edif', 'n_edif_fuera', 'm2_edif', 'm2_fuera']).get('sum').getInfo()
print(f"Edificaciones: {tot[0]:,.0f} · fuera de predio catastrado: {tot[1]:,.0f} ({tot[1]/max(tot[0],1):.1%})")
print(f"m² construidos fuera de catastro: {tot[3]:,.0f} de {tot[2]:,.0f} ({tot[3]/max(tot[2],1):.1%})")


## 11. Exportación de Resultados
- **Público (geovisor y atlas):** indicadores agregados por unidad y la mancha GHSL, en GeoJSON.
- **Uso interno:** ocupación por lote (CSV **sin geometría**), solo para el equipo técnico y la AMDC.

Las tareas aparecen en la pestaña *Tasks* de Earth Engine y se guardan en la carpeta de Drive `Copernicus_BrechaCatastral`.

**Dos etapas.** Con `CONFIG['etapa'] = 1` se exporta la imagen base de 4 m a los assets del proyecto (puede tardar una o más horas). Cuando termine, cambiar a `2` y volver a ejecutar desde la sección 3: las celdas de 250 m se agregan sobre esa imagen en pocos minutos.


In [ ]:
CAMPOS_PUBLICOS = ['unidad', 'SECTOR', 'MANZANA', 'NOM_COL', 'ha', 'n_edif', 'n_edif_fuera', 'n_lotes',
                   'm2_edif', 'm2_fuera', 'dens_edif_ha', 'area_media_m2', 'cobertura_edif', 'p_nuevo',
                   'p_critico', 'cobertura_catastral', 'p_fuera_catastro', 'brecha_catastral',
                   'tipologia', 'tipologia_brecha', 'm2_valido']
CARPETA = 'Copernicus_BrechaCatastral'

# Earth Engine corta por tiempo las exportaciones muy grandes: se dividen en franjas (celdas)
# o en grupos de sectores (manzanas, lotes). Los archivos se unen después para el geovisor.
GRUPOS = [['01', '02', '03', '04', '05', '06', '08', '10', '11', '14', '15'],
          ['07', '09'], ['13'], ['16', '17', '18', '22'], ['19', '20'],
          ['21', '24'], ['23', '26', '27'], ['25', '28']]
TODOS = [s for g in GRUPOS for s in g]
FILTROS_SECTOR = [ee.Filter.inList('SECTOR', g) for g in GRUPOS] + [ee.Filter.inList('SECTOR', TODOS).Not()]

tareas = []
if CONFIG['etapa'] == 1:
    # Imagen base de 4 m → asset privado (insumo de la etapa 2)
    tareas.append(ee.batch.Export.image.toAsset(
        image=resultado['base'], description='MDC_base_4m', assetId=CONFIG['assetBase'],
        region=aoi, scale=4, crs=CONFIG['epsg'], maxPixels=1e11,
        pyramidingPolicy={'.default': 'sample'}))
    # Público: mancha urbana consolidada GHSL → geovisor/capas/mdc_mancha_urbana.geojson
    tareas.append(ee.batch.Export.table.toDrive(
        collection=ee.FeatureCollection([ee.Feature(mancha, {'nombre': 'Mancha urbana consolidada GHSL 2020',
                                                              'umbral_m2_ha': CONFIG['umbralManchaM2']})]),
        description='MDC_mancha_urbana_ghsl', folder=CARPETA, fileFormat='GeoJSON'))

# Público: brecha agregada → geovisor/capas/mdc_brecha_agregada.geojson (etapa 2)
uni = resultado['unidades']
if CONFIG['etapa'] != 2:
    partes = []
elif CONFIG['unidad'] == 'celda':
    cortes = CONFIG['cortesCelda']
    partes = [(f's{k + 1}', uni.filter(ee.Filter.rangeContains('cx', cortes[k], cortes[k + 1])))
              for k in range(len(cortes) - 1)]
else:
    partes = [(f'g{k + 1}', uni.filter(f)) for k, f in enumerate(FILTROS_SECTOR)]
for sufijo, sub in partes:
    tareas.append(ee.batch.Export.table.toDrive(
        collection=resultado['indicadores'](sub).filter(ee.Filter.gt('m2_valido', 100)).select(CAMPOS_PUBLICOS),
        description=f"MDC_brecha_{CONFIG['unidad']}_{sufijo}", folder=CARPETA, fileFormat='GeoJSON'))

# USO INTERNO: ocupación por lote, sin geometría (activar solo si se necesita)
EXPORTAR_LOTES = False
for k, f in enumerate(FILTROS_SECTOR if (EXPORTAR_LOTES and CONFIG['etapa'] == 2) else []):
    tareas.append(ee.batch.Export.table.toDrive(
        collection=resultado['lotes'](f), description=f'MDC_ocupacion_lotes_USO_INTERNO_g{k + 1}',
        folder=CARPETA, fileFormat='CSV',
        selectors=['CLAVE_CAT', 'SECTOR', 'MANZANA', 'LOTE', 'COD_COL', 'm2_edif', 'p_ocupado', 'ocupado']))

# Descomenta para lanzar las exportaciones:
# for t in tareas: t.start()
print(f"✔ {len(tareas)} exportaciones configuradas (descomenta el bucle para lanzarlas).")
